In [ ]:
upstream = None
product = None
tree = None
tree_metadata = None
motus_path = None
motus_file = None
sanchis_tables1 = None


# MIRIPVIR25 - Draft 00 - Data processing

In [3]:
import pandas as pd
import requests
import json
import seaborn as sns 
import datetime
import random

In [4]:
def timestamp_code():
    """
    This function generates a time stamp that goes
    to microsecond level, and it adds a random code
    just in case
    """
    now = datetime.datetime.now()
    timestamp = now.strftime("%Y%m%d%H%M%S%f")
    random.seed(timestamp[-4:])
    random_code = "{:03d}".format(random.randint(0, 100))
    return timestamp +  random_code

print(timestamp_code())
print(timestamp_code())
for i in range(1000):
    if timestamp_code() == timestamp_code():
        raise RuntimeError("the function did not work")

20251006112758320172003
20251006112758320413053


In [ ]:
index_df = pd.read_csv(motus_path + "/" + motus_file, sep=",", header=None, index_col=0, names=['library', 'read1', 'read2'])
index_df

,library,read1,read2
1,PV001,PV1_18807_ATCACG_read1.fastq.gz,PV1_18807_ATCACG_read2.fastq.gz
2,PV002,PV2_18808_TGACCA_read1.fastq.gz,PV2_18808_TGACCA_read2.fastq.gz
3,PV003,PV3_16716_GGCTAC_read1.fastq.gz,PV3_16716_GGCTAC_read2.fastq.gz
4,PV003bgi,FCH5VT5ALXX_L5_HKRDPLAgshTAACRAAPEI-203_1.fq.zip,FCH5VT5ALXX_L5_HKRDPLAgshTAACRAAPEI-203_2.fq.zip
5,PV004bgi,FCH5VT5ALXX_L5_HKRDPLAgshTAADRAAPEI-204_1.fq.zip,FCH5VT5ALXX_L5_HKRDPLAgshTAADRAAPEI-204_2.fq.zip
6,PV005bgi,FCH5VT5ALXX_L5_HKRDPLAgshTAAERAAPEI-205_1.fq.zip,FCH5VT5ALXX_L5_HKRDPLAgshTAAERAAPEI-205_2.fq.zip
7,PV006bgi,FCH5VT5ALXX_L5_HKRDPLAgshTAAFRAAPEI-206_1.fq.zip,FCH5VT5ALXX_L5_HKRDPLAgshTAAFRAAPEI-206_2.fq.zip
8,PV007bgi,FCH5VT5ALXX_L5_HKRDPLAgshTAAGRAAPEI-207_1.fq.zip,FCH5VT5ALXX_L5_HKRDPLAgshTAAGRAAPEI-207_2.fq.zip
9,PV008bgi,FCH5VT5ALXX_L5_HKRDPLAgshTAAHRAAPEI-208_1.fq.zip,FCH5VT5ALXX_L5_HKRDPLAgshTAAHRAAPEI-208_2.fq.zip
10,PV009,PV9_16717_GTCCGC_read1.fastq.gz,PV9_16717_GTCCGC_read2.fastq.gz


In [7]:
def load_motus_classification(x, library):
    try:
        u = pd.read_csv(x, sep="\t")
    except pd.errors.EmptyDataError:
        u = pd.DataFrame(data=[], columns=['mOTU', 'taxonomy', 'taxid', 'abundance'])
    u.columns = ['mOTU', 'taxonomy', 'taxid', 'abundance']
    u = u.dropna(subset=['taxid'])
    u['library'] = library
    return u

In [ ]:
bacteria = dict()
hits = []
for _, row in index_df.iterrows():
    
    u = load_motus_classification("{:s}/{:s}.classification.csv".format(motus_path, row['library']), row['library'])
    
    if len(u) > 0:
        for _, item in u.iterrows():
            if item.taxid not in bacteria.keys():
                
                bacteria[item.taxid] = timestamp_code()
            taxon_code = bacteria[item.taxid]
            hits.append(
                {
                    'Hit_code': timestamp_code(),
                    'Library': item.library,
                    'Taxon_code': taxon_code
                    
                }
            )


In [9]:
pd.DataFrame.from_records(hits)

,Hit_code,Library,Taxon_code
0,20251006112904721554077,PV001,20251006112904721449075
1,20251006112904724839100,PV002,20251006112904724762032
2,20251006112904725020018,PV002,20251006112904721449075
3,20251006112904725256079,PV002,20251006112904725189069
4,20251006112904728599082,PV003,20251006112904728537097
...,...,...,...
162,20251006112904868042007,PV050,20251006112904721449075
163,20251006112904870695087,PV051,20251006112904724762032
164,20251006112904873507003,PV052,20251006112904724762032
165,20251006112904876113085,PV053,20251006112904724762032


In [10]:
bacteria = pd.DataFrame(list(bacteria.items()), columns=['taxonid', 'Taxon_code'])
bacteria['taxonid'] = bacteria['taxonid'].astype(int)

In [11]:
import taxoniq

def fault_tolerant_name(x):
    try:
        return taxoniq.Taxon(x).scientific_name
    except KeyError:
        return ""

In [12]:
bacteria['scientific_name'] = bacteria['taxonid'].apply(lambda x: fault_tolerant_name(x))

In [13]:
bacteria

,taxonid,Taxon_code,scientific_name
0,2097,20251006112904721449075,Mycoplasmoides genitalium
1,59620,20251006112904724762032,uncultured Clostridium sp.
2,69896,20251006112904725189069,Candidatus Phytoplasma solani
3,641148,20251006112904728537097,Neisseria sp. oral taxon 014
4,1236179,20251006112904729024030,Janthinobacterium sp. B9-8
...,...,...,...
67,553,20251006112904856853045,Pantoea ananatis
68,1050843,20251006112904857386064,
69,1528099,20251006112904857596017,Lawsonella clevelandensis
70,1290,20251006112904863901026,Staphylococcus hominis


In [ ]:
from skbio.tree import TreeNode
tree = TreeNode.read(tree)
gtdb_metadata = pd.read_csv(tree_metadata, sep="\t")
tree_tip_names = list(tree.subset())
gtdb_metadata[gtdb_metadata['gtdb_genome_representative'].apply(lambda x: x.replace("_", " ")).isin(tree_tip_names)]
gtdb_accessions = gtdb_metadata[['ncbi_taxid', 'gtdb_genome_representative']].drop_duplicates('ncbi_taxid', keep='first').copy()
gtdb_accessions['gtdb_genome_representative'] = gtdb_accessions['gtdb_genome_representative'].apply(lambda x: x.replace("_", " "))
# gtdb_accessions.to_csv('scratch/semantic-model/gtdb.csv', sep=',')
gtdb_accessions

,ncbi_taxid,gtdb_genome_representative
0,1331258,RS GCF 000657795.2
1,1282,RS GCF 006742205.1
2,2135698,RS GCF 000172415.1
3,90371,RS GCF 000006945.2
4,1236944,RS GCF 029023865.1
...,...,...
584333,1752224,GB GCA 001464955.1
584341,2665509,RS GCF 000006945.2
584345,2530375,RS GCF 004348725.1
584349,2169409,RS GCF 003097655.1


In [17]:
bacteria = pd.merge(bacteria, gtdb_accessions, left_on='taxonid', right_on='ncbi_taxid', how='left')

In [ ]:
sanchis21 = pd.read_csv(sanchis_tables1, sep="\t").drop_duplicates('TaxId', keep='first')
sanchis21['is_pab'] = True
sanchis21['pab_type'] = sanchis21.apply(lambda x: 'pab_pathogen' if x['PAB-phyto'] == 'P' else "pab_unknown", axis=1)
sanchis21['pab_type'] = sanchis21.apply(lambda x: 'pab_symbiont' if x['PAB-symb'] == 'S' else x['pab_type'], axis=1)
sanchis21

,TaxId,Biosample,Representative species,PAB-phyto,PAB-symb,is_pab,pab_type
0,178901,SAMN02849420,Acetobacter malorum,NaN,NaN,True,pab_unknown
1,441768,SAMN02603756,Acholeplasma laidlawii PG-8A,NaN,NaN,True,pab_unknown
2,32002,SAMN06198582,Achromobacter denitrificans,NaN,NaN,True,pab_unknown
3,217204,SAMN06174288,Achromobacter insolitus,NaN,NaN,True,pab_unknown
4,72556,SAMN03941583,Achromobacter piechaudii,NaN,NaN,True,pab_unknown
...,...,...,...,...,...,...,...
955,698414,SAMN06765826,Xylella fastidiosa subsp. pauca,P,NaN,True,pab_pathogen
956,1444770,SAMN02570451,Xylella taiwanensis,P,NaN,True,pab_pathogen
957,1735686,SAMN04151686,Xylophilus sp. Leaf220,NaN,NaN,True,pab_unknown
958,1288385,SAMEA1486427,Yersinia pekkanenii,NaN,NaN,True,pab_unknown


In [21]:
bacteria = pd.merge(bacteria, sanchis21[['TaxId', 'is_pab', 'pab_type']], left_on=['ncbi_taxid'], right_on=['TaxId'], how='left')

In [22]:
bacteria

,taxonid,Taxon_code,scientific_name,ncbi_taxid,gtdb_genome_representative,TaxId,is_pab,pab_type
0,2097,20251006112904721449075,Mycoplasmoides genitalium,NaN,NaN,NaN,NaN,NaN
1,59620,20251006112904724762032,uncultured Clostridium sp.,59620.0,GB GCA 900548855.1,NaN,NaN,NaN
2,69896,20251006112904725189069,Candidatus Phytoplasma solani,69896.0,RS GCF 000970375.1,NaN,NaN,NaN
3,641148,20251006112904728537097,Neisseria sp. oral taxon 014,641148.0,RS GCF 005886145.1,NaN,NaN,NaN
4,1236179,20251006112904729024030,Janthinobacterium sp. B9-8,1236179.0,RS GCF 000969645.2,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
67,553,20251006112904856853045,Pantoea ananatis,553.0,RS GCF 000710035.2,553.0,True,pab_pathogen
68,1050843,20251006112904857386064,,NaN,NaN,NaN,NaN,NaN
69,1528099,20251006112904857596017,Lawsonella clevelandensis,1528099.0,RS GCF 001293125.1,NaN,NaN,NaN
70,1290,20251006112904863901026,Staphylococcus hominis,1290.0,RS GCF 002901845.1,NaN,NaN,NaN


In [23]:
def clean_dict(x):
    new_dict = dict()
    for key, item in x.items():
        if not pd.isna(item):
            new_dict[key] = item
    return new_dict

In [ ]:
bacteria_records = dict(
    taxon=list(map(clean_dict, bacteria.to_dict(orient='records'))),
    hits=hits
)

with open(product['data'], "w") as f:
    json.dump(bacteria_records, f, indent=4, sort_keys=True)